<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<img src="../media/img/kisz_logo.png" width="192" height="69"> 
<h1> Working with embeddings:
<h2>An introductory workshop with applications on Semantic Search
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part 3.2 - Word2Vec from scratch
</div>

In this section, we will build a Word2Vec model from scratch, exploring both the Continuous Bag of Words (CBOW) and Skip-gram architectures in detail. Unlike the previous notebook, where we used <kbd>gensim</kbd>, here we will implement the training process manually using NumPy. We will cover data preprocessing, model training, and optimization techniques such as negative sampling. Finally, we will analyze the learned embeddings and evaluate their effectiveness in capturing semantic relationships between words.

We start as usual importing some packages.

In [ ]:
# imports
import sys
import pandas as pd
import numpy as np

from nb_config import INTERIM_DATA_PATH

from src.data import data_loader

import warnings
warnings.filterwarnings('ignore')

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Overview
</div>

We have seen in the last notebook that Word2Vec is a neural network-based approach for learning word embeddings that capture semantic relationships between words. In this notebook, we will implement Word2Vec from scratch, covering both the Continuous Bag of Words (CBOW) and Skip-gram architectures in depth.

Let's talk about this two architectures:

- Continuous Bag of Words (CBOW):
    - The CBOW model predicts a target word based on a given context (surrounding words).
    - Given a window of words around the target word, the model learns to predict the target word using these context words.
    - The model is trained by adjusting the word vectors such that words appearing in similar contexts have similar embeddings.
    - CBOW is generally faster and works well with smaller datasets but may not capture rare words as effectively as Skip-gram.

- Skip-gram
  - The Skip-gram model does the reverse of CBOW: it predicts the surrounding words given a target word.
  - It learns to maximize the probability of correctly predicting context words for a given target word.
  - Skip-gram is more computationally expensive but performs well on large datasets and captures rare words better than CBOW.

The training process involves the following steps:

- Preparing the Data
  - Tokenizing text into words and constructing a vocabulary.
  - Creating training pairs (context-target for CBOW or target-context for Skip-gram).
- Building the Neural Network
  - A simple neural network with an input layer, a hidden layer (embedding layer), and an output layer.
  - The hidden layer learns word embeddings, which are adjusted during training.
- Training the Model
  - Using one-hot encoding to represent words in the vocabulary.
  - Using softmax activation in the output layer to predict the target/context words.
  - Employing backpropagation and stochastic gradient descent (SGD) to optimize word vectors.
  - To improve efficiency, negative sampling or hierarchical softmax can be used.
- Extracting Word Embeddings
  - Once training is complete, the word vectors from the hidden layer can be used for semantic analysis.
  - Similar words should have close vector representations in the learned space.

As the way of implementing both architectures is quite similar we will do it at the same time, but taking a bit more of time in the parts that are different.

We are going to need at this point our tokens. Let's load them.

In [ ]:
df, params = data_loader("my_tokenized_data.parquet")

# extract the arguments for the normalize function
tokenizer = params['tokenizer']
arguments = params['args']

# print the tokenizer used getting the tokens
print(f"Tokenizer used: {tokenizer}")
print(f"Arguments passed to the normalize function:\n{arguments}")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Preparing the data
</div>

We are going to create our vocabulary and associate every token to a specific index. We need to be able to map every token to its index and vice versa.

In [ ]:
# SOLUTION

# create an easy access to the tokens
tokens = df['tokens']

# create an empty set
vocabulary = set()

# iterate over the pd Series and updates the set
for index in df.index:
   vocabulary.update(df.loc[index, 'tokens'].tolist())

# turn the vocabulary into a list
vocabulary = list(vocabulary)

# print vocabulary size
vocab_size = len(vocabulary)

print(f"The vocabulary has {vocab_size} tokens.")

We are going to create now two mappings for the tokens and their indexes, one for each direction. That will make things much easier

In [ ]:
# SOLUTION
token_to_idx = {token: idx for idx, token in enumerate(vocabulary)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}

We need now to create the training pairs.

For that we will create a empty list and will populate it with tuples. Each tuple corresponds to a single training datapoint and is composed of two elements:
- the central word
- the context words

In CBoW the context words will be the input and the central word the expected output. In the SkipGram architecture we have the inverse case, the central word as input and the context words as expected ouputs.

We start by formatting our corpus. Our texts have already been tokenized, so we can create a tokenized version of the corpus, where we bring into a list the list of tokens for each film.

In [ ]:
## SOLUTION

tk_corpus = tokens.tolist()

# define the function
def generate_training_data(corpus, window_size=2):
    training_data = []
    for doc in corpus:
        for i in range(len(doc)):
            context = np.append(doc[max(0, i - window_size):i], doc[i + 1:i + 1 + window_size])
            center = doc[i]
            training_data.append((center, context))
    return training_data

training_data = generate_training_data(tk_corpus)

We can take a look and see how does it look like:

In [ ]:
training_data

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Building the neural network
</div>

Our next step is building the Neural Network. Both architectures are different but still share some common patterns. Let's discuss it with more detail.

Abbreviations used:
- **$B$**: Batch size
- **$V$**: Vocabulary size
- **$S$**: Embeddings size
- **$W_i$**: Weight matrices

#### 3.1 CBoW

The CBoW model follows a shallow, two-layer neural network design:

1. **Input Layer**
   
   The input for every training data point consists of a sparse vector of size $V$ for each context token surrounding the target token. Those vectors are obtained by one hot encoding the tokens using the vocabulary, and together they are represented by a matrix of size of at most $2C \times V$. As they basically work as a lookup table, we can add them together to get a sparse vector of size $V$ with so many ones as context tokens, if not token is repeated.
      
2. **Projection Layer (Hidden Layer)**

    The hidden layer consists of a single dense layer with a weight matrix $W_1$ of shape $V \times N$.
    
    For each data point, the input vector is projected into this dense layer, and their embeddings are **averaged** to form a single vector representation.

3. **Output Layer**

    The output layer is a softmax classifier that predicts the target token.
    It contains a second weight matrix $W_2$ of shape $N \times V$.
    The output is a probability distribution over the vocabulary, indicating the likelihood of each word being the correct target.

Wait, why do we average instead of just adding?

- **Preventing Magnitude Issues**: If we sum the word embeddings, the resulting vector's magnitude depends on the number of context words. Larger context windows would lead to larger summed values, making training unstable. Averaging normalizes this effect, ensuring that all context sizes contribute equally.
- **Consistency Across Different Context Sizes**: If we sum, a sentence with a larger context window would have a significantly different vector scale compared to a sentence with a smaller context window. Averaging ensures that the model learns embeddings that are scale-invariant to the number of context words.
- **Smoother Optimization & Stable Learning**: Gradient updates become more stable with averaging. If we sum, the magnitude of gradients would vary based on the number of context words, leading to inconsistent learning rates. Averaging keeps gradients on a consistent scale, making training more efficient.

##### 3.1.1 Writing the class

Let's start putting everything together. We will start defining a class called <kbd>Word2VecCBOW</kbd> and write its initialization function. We will need to pass some parameters when we instatiate the class:
- **vocab_size**
- **embedding_dim**
- **learning_rate** (with a default value of 0.01)

We will also initialize our weights matrices W1 and W2 with <kbd>np.random.uniform</kbd> with values between -1 and 1.

In [ ]:
## SOLUTION
class Word2VecCBOW:
    def __init__(self, vocab_size, embedding_dim, learning_rate=0.01):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.learning_rate = learning_rate
        
        # Initialize weight matrices
        self.W1 = np.random.uniform(-1, 1, (vocab_size, embedding_dim))  # Input to hidden
        self.W2 = np.random.uniform(-1, 1, (embedding_dim, vocab_size))  # Hidden to output


Oh! Only weights? Why there is no Bias vector in CBoW?

There are a few reasons for it:

- **Simplicity**: Adding a bias term doesn’t significantly improve the embeddings, so it's often omitted to keep the model lightweight.
- **Embedding Interpretation**: The key part of training Word2Vec is learning the weight matrix $W$, which directly maps words to dense vector representations. Bias terms would shift the embeddings, but since embeddings are used in similarity computations (dot product or cosine similarity), shifting them doesn't add much value.
- **Averaging Mechanism**: Since CBoW averages the input word vectors before passing them to the softmax layer, a bias term wouldn’t have the same impact as in standard neural networks. The model mainly relies on word co-occurrence patterns, and the averaged embeddings already provide a useful contextual signal.



##### 3.1.2 The input layer

We prepare now the input vectors.

Since the input (one-hot encoding) vectors effectively act as a lookup table for the weight matrix $W_1$​, selecting only specific rows, we can bypass the explicit creation of one-hot vectors. Instead, we'll directly use their indices to access the corresponding rows in $W_1$​ using NumPy's fancy indexing.

In [ ]:
training_idx = [[token_to_idx[item[0]], [token_to_idx[token] for token in item[1]]] for item in training_data]

In [ ]:
training_idx

##### 3.1.3 Forward Pass

In this step the input vectors are multiplied by the embedding matrix $W_1$ and averaged. The averaged vector is then passed through the output weight matrix $W_2$ and a softmax activation is applied to produce a probability distribution over all words in the vocabulary.

Let's write the function that will do that for us.


In [ ]:
## SOLUTION
def forward(self, context):
        h = np.mean(self.W1[[context]], axis=1)  # Shape: (1, embedding_dim)
        
        # Compute output scores
        u = np.dot(h, self.W2)  # Shape: (1, vocab_size)
        
        # Softmax activation
        y_pred = np.exp(u) / np.sum(np.exp(u))

        return h, y_pred

setattr(Word2VecCBOW, 'forward', forward)

##### 3.1.4 Loss and Backpropagation

We want to train the model using Cross-Entropy Loss. So, given our output

$\hat{y}_i = \frac{e^{u_i}}{\sum_j e^{u_j}}$

the Cross-Entropy-Loss is

$L = -\sum\limits_{i \in center} y_i \cdot log (\hat{y}_i)$

The gradient with respect to the pre-softmax scores $u$ (logits) is then

$\frac{\partial L}{\partial u} = \hat{y} - y$

That leads to

$\frac{\partial L}{\partial W_2} = h^\intercal (\hat{y} - y)$,

$\frac{\partial L}{\partial h} = (\hat{y} - y) W_2^\intercal$,

and

$\frac{\partial L}{\partial W_1} = \frac{1}{N} (\hat{y} - y) W_2^\intercal$

for those rows corresponding to the tokens in the context, with N the context size.




In [ ]:
## SOLUTION
def backward(self, context, target, h, y_pred):
        # Compute error (difference between prediction and target one-hot vector)
        error = y_pred - np.eye(vocab_size)[target]  # Shape: (1, vocab_size)

        # Gradients for W2
        dW2 = np.dot(h.T, error)  # Shape: (embedding_dim, vocab_size)

        # Gradients for W1
        dW1 = np.dot(error, self.W2.T) / len(context)  # Shape: (1, embedding_dim)
                                                        # for every token in context

        # Update weights
        self.W1[[context]] -= self.learning_rate * dW1
        self.W2 -= self.learning_rate * dW2

setattr(Word2VecCBOW, 'backward', backward)

##### 3.1.5 Training loop

We need now a function that puts everything together and loops over all the training datapoints.

In [ ]:
def train(self, training_idx, epochs=1000):
    for epoch in range(epochs):
        loss = 0
        for target, context in training_idx:
            # Forward pass
            h, y_pred = self.forward(context)
            
            # Compute loss (Cross-Entropy)
            loss -= np.log(y_pred[0, target])
            
            # Backward pass
            self.backward(context, target, h, y_pred)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

setattr(Word2VecCBOW, 'train', train)

Let's see if this works.

In [ ]:
model = Word2VecCBOW(vocab_size, embedding_dim=10, learning_rate=0.01)
model.train(training_idx[:5], epochs=1000)

The weight matrix $W_1$ contains our embeddings!